# Task 3 — Usage U2 full-RGB HOG linear SVM

Run All performs one fixed two-fold CPU screen. It uses a pure full-image RGB HOG descriptor, weighted scaling, a one-vs-rest linear SVM with `C=1`, and sigmoid calibration. There is no feature grid or model search.


## 1. Local runtime

Use the repository `.venv` kernel. This fixed-feature model runs on CPU and writes to the gitignored local Task 3 artifact folder.


In [1]:
from pathlib import Path
import os
import subprocess
import sys

def find_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/fashion").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the cloned repository.")

REPO_DIR = find_repo_root()
expected_venv = (REPO_DIR / ".venv").resolve()
if Path(sys.prefix).resolve() != expected_venv:
    raise RuntimeError(f"Select the repository .venv kernel: {expected_venv}")
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

LOCAL_TASK_DIR = REPO_DIR / "results/task3"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"
LOCAL_EVIDENCE_DIR = REPO_DIR / "results/evidence/task3"
LOCAL_WORK_DIR = REPO_DIR / "tmp/task3-usage-hog"
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


Branch: task-3-gender-usage-classification
Commit: 05a523f257e15575859f57608d08c4ff351185d5


In [2]:
teacher_dir = REPO_DIR / "data/raw/teacher"
required = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
    REPO_DIR / "data/processed/splits.csv",
    REPO_DIR / "data/processed/label_maps.json",
)
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Required teacher/processed files are missing: {missing}")
print("Official teacher data and canonical split are ready.")


Official teacher data and canonical split are ready.


## 2. Frozen hypothesis and gates

EDA found that pure full-RGB HOG was the strongest direct fixed-image probe for Usage (balanced diagnostic macro-F1 about 0.454). U2 asks whether replacing E2’s learned CNN representation with this lower-freedom shape/texture representation reduces memorisation while keeping rare-class signal. Labels, canonical folds, full canvas, effective-number weighting, and the E2 comparison rows stay fixed. No augmentation is used.

The screen advances by Route A only if macro-F1 is at least 0.417319 and the paired family-bootstrap lower 95% bound is above zero. Route B needs macro-F1 at least 0.402319 plus at least 25% clean-gap reduction or at least 10% NLL/Brier improvement. ECE must be at most 0.050, no supported class may lose more than 0.030 F1, and probability/split/resource checks must pass.


Weights are computed from each inner base-training subset and reach only its scaler and SVM. Sigmoid calibration uses the natural, unweighted held-out fold. All eight non-Home classes must occur in every inner training and calibration subset. Home is excluded from fitting and calibration, has probability zero, and stays in the official nine-class score (including its NLL penalty).

The four rare classes are NA, Party, Smart Casual, and Travel. Each prediction count must be at most five times its true support, both pooled and per fold. Each of the five saved corruption tests must lose no more than 0.020 extra macro-F1 compared with E2. Every fold must finish, including diagnostics, within 90 minutes and 7 GiB. Missing evidence cannot pass. E2 has no comparable saved clean finished-model training score, so that Route B branch is unavailable; its NLL/Brier branch remains usable.


In [3]:
USAGE_E2_PARENT_RUN_IDS = (
    "t3_usage_e2_class_balanced_ce_usage_smallcnn_f0_s2753_5461e048c3b3_20260830T115815Z356f6d",
    "t3_usage_e2_class_balanced_ce_usage_smallcnn_f1_s2753_5461e048c3b3_20260830T120645Z6d08cd",
    "t3_usage_e2_class_balanced_ce_usage_smallcnn_f2_s2753_5461e048c3b3_20260830T121514Zd58aa0",
    "t3_usage_e2_class_balanced_ce_usage_smallcnn_f3_s2753_5461e048c3b3_20260830T122347Z2cb06e",
    "t3_usage_e2_class_balanced_ce_usage_smallcnn_f4_s2753_5461e048c3b3_20260830T123218Z94db47",
)
USAGE_E2_ANCHOR = (
    LOCAL_EVIDENCE_DIR
    / "experiments/t3_usage_e2_class_balanced_ce/usage/aggregate/oof_predictions.csv"
)
if not USAGE_E2_ANCHOR.is_file():
    raise FileNotFoundError(
        f"Sync the E2 evidence first; the matched anchor is missing: {USAGE_E2_ANCHOR}"
    )

from fashion.train.task3_usage_hog_svm import check_usage_hog_svm_setup
USAGE_E2_REGISTRY = LOCAL_EVIDENCE_DIR / "results/runs.csv"
preflight = check_usage_hog_svm_setup(
    root=REPO_DIR, folds=(0, 4), parent_run_ids=USAGE_E2_PARENT_RUN_IDS,
    anchor_prediction_path=USAGE_E2_ANCHOR, parent_registry_path=USAGE_E2_REGISTRY,
)
if not preflight["ready"]:
    raise RuntimeError(preflight["training_blockers"])
print("Parent evidence:", preflight["parent_evidence"])
print("Model:", preflight["model"])
print("Screen folds:", preflight["folds"])
print("Feature columns:", preflight["feature_columns"])
print("Estimated cache MiB:", round(preflight["estimated_cache_bytes"] / 1024**2, 1))
print("Model fits during check:", preflight["model_fits"])


Parent evidence: verified
Model: full-RGB HOG + weighted StandardScaler + calibrated LinearSVC(C=1)
Screen folds: [0, 4]
Feature columns: 1944
Estimated cache MiB: 243.0
Model fits during check: 0


## 3. Build or reuse the label-blind HOG cache

This is the long preparation step. It reads only official teacher images and does not fit a model.


In [4]:
from fashion.train.task3_usage_hog_svm import prepare_usage_hog_features

prepared_features = prepare_usage_hog_features(
    root=REPO_DIR,
    output_root=LOCAL_TASK_DIR,
    workers=None,
    local_work_dir=LOCAL_WORK_DIR,
)
print(prepared_features["usage"])


[task3-clean-slate] building feature view=full_rgb_hog on local disk: /home/dinhquan/personal/academic/RMIT/Machine-Learning/MLA2-eda/tmp/task3-usage-hog/full_rgb_hog_74cb33cdb6d3e0e0_8te_wnig.npy
[task3-clean-slate] feature view=full_rgb_hog: 2,500/32,773 images
[task3-clean-slate] feature view=full_rgb_hog: 5,000/32,773 images
[task3-clean-slate] feature view=full_rgb_hog: 7,500/32,773 images
[task3-clean-slate] feature view=full_rgb_hog: 10,000/32,773 images
[task3-clean-slate] feature view=full_rgb_hog: 12,500/32,773 images
[task3-clean-slate] feature view=full_rgb_hog: 15,000/32,773 images
[task3-clean-slate] feature view=full_rgb_hog: 17,500/32,773 images
[task3-clean-slate] feature view=full_rgb_hog: 20,000/32,773 images
[task3-clean-slate] feature view=full_rgb_hog: 22,500/32,773 images
[task3-clean-slate] feature view=full_rgb_hog: 25,000/32,773 images
[task3-clean-slate] feature view=full_rgb_hog: 27,500/32,773 images
[task3-clean-slate] feature view=full_rgb_hog: 30,000/32,7

## 4. Train folds 0 and 4

A matching completed fold is reused. Stop after this screen and analyse it before any five-fold run.

Each fold runs in a fresh supervised process. The parent stops it at 90 minutes, including startup, fitting, corruption checks, and artifact work. The worker also has a hard 7 GiB address-space cap. This counts memory mappings as well as RAM, so it is stricter than the original peak-RAM guard. A stopped worker leaves a failed registry row and cannot be reused. Only the parent marks a run complete after successful worker exit.

The solver uses one numerical thread. Iteration counts, convergence flags, and warning messages are saved in `solver_history.csv`. The updated resource contract prevents reuse of older unsupervised runs.


In [5]:
from fashion.train.task3_usage_hog_svm import run_usage_hog_svm_screen

usage_u2 = run_usage_hog_svm_screen(
    prepared_features=prepared_features,
    parent_run_ids=USAGE_E2_PARENT_RUN_IDS,
    output_root=LOCAL_TASK_DIR,
    folds=(0, 4),
    registry_path=LOCAL_REGISTRY,
    root=REPO_DIR,
    anchor_prediction_path=USAGE_E2_ANCHOR,
    parent_registry_path=USAGE_E2_REGISTRY,
    reuse_completed=True,
)
{
    "metrics_path": usage_u2["metrics_path"],
    "macro_f1": usage_u2["metrics"]["macro_f1"],
    "nll": usage_u2["metrics"]["nll"],
    "brier": usage_u2["metrics"]["brier"],
    "ece_15": usage_u2["metrics"]["ece_15"],
    "screen_gate": usage_u2["metrics"]["screen_gate"],
}


[task3-usage-hog] fold=0: fitting 4 calibrated SVMs
[task3-usage-hog] jpeg_75: macro-F1=0.307474
[task3-usage-hog] brightness_085: macro-F1=0.381711
[task3-usage-hog] brightness_115: macro-F1=0.325067
[task3-usage-hog] translation_003: macro-F1=0.249291
[task3-usage-hog] grayscale: macro-F1=0.393072
[task3-usage-hog] fold=4: fitting 4 calibrated SVMs
[task3-usage-hog] jpeg_75: macro-F1=0.295639
[task3-usage-hog] brightness_085: macro-F1=0.351639
[task3-usage-hog] brightness_115: macro-F1=0.260298
[task3-usage-hog] translation_003: macro-F1=0.238435
[task3-usage-hog] grayscale: macro-F1=0.337813


{'metrics_path': '/home/dinhquan/personal/academic/RMIT/Machine-Learning/MLA2-eda/results/task3/experiments/t3_usage_v2_u2_full_rgb_hog_svm/usage/aggregate_folds_0_4/metrics.json',
 'macro_f1': 0.36293188251462905,
 'nll': 0.3997488067999392,
 'brier': 0.20736386299327048,
 'ece_15': 0.03210095854484196,
 'screen_gate': {'status': 'fail',
  'gate_version': 'u2_routes_natural_calibration_v2',
  'checks': [{'gate': 'canonical_oof_integrity',
    'value': 13110,
    'rule': 'exact canonical folds 0 and 4',
    'status': 'pass'},
   {'gate': 'unsupported_home_zero',
    'value': True,
    'rule': 'P(Home)=0 on every row',
    'status': 'pass'},
   {'gate': 'ece_15',
    'value': 0.03210095854484196,
    'rule': '<= 0.050',
    'status': 'pass'},
   {'gate': 'rare_prediction_cap',
    'value': [{'scope': 'pooled',
      'class_name': 'NA',
      'support': 22,
      'predicted_count': 3,
      'maximum': 110,
      'pass': True},
     {'scope': 'pooled',
      'class_name': 'Party',
      '

## 5. Stop

Do not train a second Usage model from this notebook.
